# 01 — Pulse Generation and Chirp Intuition

This notebook shows how a plain rectangular pulse becomes a frequency-swept LFM chirp, and why that sweep gives a radar much better range resolution than the pulse length alone would allow.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/radar-tracker-notebooks/beginner/01-pulse-chirp-intuition.ipynb)

Use this link if you want to open the notebook in Colab and follow along in a browser notebook environment.

## What this notebook teaches

In Notebook 00 we treated the transmit burst as a simple time interval. Here we look inside that interval and build the actual waveform.

By the end of this notebook, you should be able to explain:

- what a rectangular pulse looks like in time,
- what an LFM chirp is and how its frequency sweeps,
- why a chirp can resolve targets much closer together than its pulse length suggests,
- and what the time-bandwidth product tells us about that gain.

Keep these four questions in mind as you work through the cells. At the end of the notebook, a dedicated section answers each one directly, so you can study the material first and then check your understanding against the full story.

## Setup and baseline values

We reuse the baseline radar specification from Notebook 00 so every waveform here is built from the same physical case. If you are running in Colab, run the bootstrap cell below first so the repository is cloned, installed, and available for import.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab starts on a fresh VM, so clone the repository and install it in editable mode
# before importing the beginner helpers.
REPO_URL = "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"
BRANCH_NAME = "radar-tracker-notebooks"
REPO_DIR = Path("/content/active-radar-tracker-basics")

in_colab = "google.colab" in sys.modules

if in_colab and not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

if in_colab:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    repo_root = os.path.abspath(".")
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    print(f"Ready in Colab from {REPO_DIR}")
else:
    repo_root = os.path.dirname(os.getcwd())
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print("Running locally; the repository is already available in this workspace.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from beginner.helpers import BaselineRadarSpec, baseline_spec
from beginner.helpers.plotting import apply_notebook_style

# Set the shared notebook styling once for consistent plots.
apply_notebook_style()

# Create one named radar specification so the lesson reads naturally.
radar_spec = BaselineRadarSpec()
radar_spec

## The rectangular pulse

The simplest transmit waveform is a rectangular pulse: the transmitter is simply on at full amplitude for the pulse width, then off. In sampled form it is just a block of ones.

The number of samples the pulse occupies is

$$
N = f_s \cdot \tau
$$

where $f_s$ is the sampling rate and $\tau$ is the pulse width. This block of energy is short in time, but its frequency content is broad only up to about $1/\tau$.

In [ ]:
# Build the rectangular pulse directly so you can see the construction.
pulse_samples = int(round(radar_spec.pulse_width_s * radar_spec.fs_hz))
rect_pulse = np.ones(pulse_samples, dtype=float)

print(f"Pulse width = {radar_spec.pulse_width_s * 1e6:.1f} microseconds")
print(f"Sampling rate = {radar_spec.fs_hz / 1e6:.1f} MHz")
print(f"Samples in one pulse N = {pulse_samples}")

{
    "pulse_samples": pulse_samples,
    "first_five": rect_pulse[:5].tolist(),
    "last_five": rect_pulse[-5:].tolist(),
}

## The LFM chirp

A linear-frequency-modulation (LFM) chirp keeps the same pulse length, but during the pulse the frequency sweeps linearly from a start value to a stop value. In the baseband form we use here, the instantaneous frequency rises from $0$ to the bandwidth $B$.

The complex baseband chirp is

$$
s(t) = \exp\!\left(j \pi \frac{B}{\tau} t^{2}\right)
$$

where the chirp rate $B/\tau$ sets how fast the frequency climbs. The key point is that the frequency range covered, $B$, can be much larger than $1/\tau$ — and that extra bandwidth is what buys us resolution later.

In [ ]:
# Build the LFM chirp directly so you can see the quadratic phase formula.
t = np.arange(pulse_samples) / radar_spec.fs_hz
chirp_rate = radar_spec.bandwidth_hz / radar_spec.pulse_width_s
chirp_phase = np.pi * chirp_rate * t**2
chirp = np.exp(1j * chirp_phase)

# Estimate the instantaneous frequency from the phase slope.
inst_freq = np.diff(np.unwrap(np.angle(chirp))) / (2.0 * np.pi) * radar_spec.fs_hz

print(f"Chirp rate = {chirp_rate:.3e} Hz/s")
print(f"Start frequency = {inst_freq[0]:.2f} Hz")
print(f"Stop frequency = {inst_freq[-1] / 1e6:.2f} MHz (approximately the bandwidth)")
print(f"Bandwidth B = {radar_spec.bandwidth_hz / 1e6:.1f} MHz")

{
    "chirp_rate": chirp_rate,
    "start_freq_hz": float(inst_freq[0]),
    "stop_freq_hz": float(inst_freq[-1]),
}

## The two waveforms side by side

Plotted in time, the rectangular pulse is flat and the chirp also looks like a constant-amplitude burst. Their difference is hidden in the phase, which is exactly why the frequency-sweep view matters next.

In [ ]:
# Show the time-domain shape of both waveforms.
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(t * 1e6, rect_pulse, color="#d95f02")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Rectangular pulse (flat in time)")

axes[1].plot(t * 1e6, np.real(chirp), color="#1b9e77", label="real part")
axes[1].plot(t * 1e6, np.imag(chirp), color="#7570b3", label="imaginary part")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlabel("Time within pulse (microseconds)")
axes[1].set_title("LFM chirp (frequency sweeps during the pulse)")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## The frequency sweep

The chirp's defining feature is its linear frequency ramp. This is the plot that separates it from the plain pulse: the plain pulse has no sweep, while the chirp climbs steadily across the whole pulse width.

In [ ]:
# Show the instantaneous frequency ramp of the chirp.
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[1:] * 1e6, inst_freq / 1e6, color="#1b9e77")
ax.axhline(radar_spec.bandwidth_hz / 1e6, color="#999999", linestyle="--", label="Bandwidth B")
ax.set_xlabel("Time within pulse (microseconds)")
ax.set_ylabel("Frequency (MHz)")
ax.set_title("Chirp instantaneous frequency sweeps 0 to B")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Why chirps matter: range resolution

Range resolution is the smallest separation between two targets that the radar can tell apart. For a plain pulse it is set by the pulse length:

$$
\Delta R_{\text{pulse}} = \frac{c \, \tau}{2}
$$

For a pulse-compressed chirp it is set by the bandwidth:

$$
\Delta R_{\text{chirp}} = \frac{c}{2 B}
$$

Because $B \gg 1/\tau$, the chirp resolves targets far closer together than the raw pulse length would allow — the long pulse keeps the energy high while the bandwidth keeps the resolution sharp.

In [ ]:
# Compare range resolution the two waveforms give, by hand.
c = 299_792_458.0
res_pulse = c * radar_spec.pulse_width_s / 2.0
res_chirp = c / (2.0 * radar_spec.bandwidth_hz)

print("Range resolution from the raw pulse length:")
print(f"  delta R_pulse = c * tau / 2 = {res_pulse:.1f} m")
print()
print("Range resolution after pulse compression by bandwidth:")
print(f"  delta R_chirp = c / (2B) = {res_chirp:.1f} m")
print()
print(f"Resolution improvement factor = {res_pulse / res_chirp:.1f}x")

{
    "res_pulse_m": res_pulse,
    "res_chirp_m": res_chirp,
    "improvement_x": res_pulse / res_chirp,
}

## The time-bandwidth product

The ratio

$$
TBP = B \cdot \tau
$$

is the time-bandwidth product. It measures how much the chirp squeezes a long pulse into a short compressed peak. A large time-bandwidth product means a long, energetic pulse can still give fine range resolution — the central trade that makes modern pulse radar practical.

In [ ]:
# Compute the time-bandwidth product directly.
tbp = radar_spec.bandwidth_hz * radar_spec.pulse_width_s

print(f"Bandwidth B = {radar_spec.bandwidth_hz / 1e6:.1f} MHz")
print(f"Pulse width tau = {radar_spec.pulse_width_s * 1e6:.1f} microseconds")
print(f"Time-bandwidth product = {tbp:.0f}")
print()
print("The pulse is long in time, but the bandwidth compresses it by this factor.")

{
    "time_bandwidth_product": tbp,
}

## Checkpoint

In your own words, why can a chirp resolve targets closer together than a plain pulse of the same length?

Then answer this: if you increase the bandwidth $B$ while keeping the pulse width $\tau$ fixed, what happens to the time-bandwidth product and to the range resolution?

## Common mistake

A common mistake is to think the pulse length alone sets resolution. It does for a plain pulse, but a chirp decouples the two: the pulse can stay long (for energy) while the bandwidth sets the resolution. That is the whole point of pulse compression.

Another mistake is to read the chirp's time-domain plot as just "a weird pulse." The resolution lives in the frequency sweep, not in the amplitude shape, so always check the frequency view before judging a waveform.

In [ ]:
# Now use the helper functions to reproduce the same waveforms and numbers compactly.
from beginner.helpers import (
    lfm_chirp,
    rectangular_pulse,
    instantaneous_frequency_hz,
    range_resolution_from_pulse_width,
    range_resolution_from_bandwidth,
)

helper_spec = baseline_spec()
helper_n = int(round(helper_spec.pulse_width_s * helper_spec.fs_hz))
helper_rect = rectangular_pulse(helper_n)
helper_chirp = lfm_chirp(helper_n, helper_spec.bandwidth_hz, helper_spec.pulse_width_s, helper_spec.fs_hz)
helper_freq = instantaneous_frequency_hz(helper_chirp, helper_spec.fs_hz)
helper_res_pulse = range_resolution_from_pulse_width(helper_spec.pulse_width_s)
helper_res_chirp = range_resolution_from_bandwidth(helper_spec.bandwidth_hz)

print("Helper-based version of the same calculations:")
print(f"  samples per pulse = {helper_n}")
print(f"  sweep 0 -> {helper_freq[-1] / 1e6:.2f} MHz")
print(f"  pulse resolution = {helper_res_pulse:.1f} m")
print(f"  chirp resolution = {helper_res_chirp:.1f} m")

{
    "samples": helper_n,
    "res_pulse_m": helper_res_pulse,
    "res_chirp_m": helper_res_chirp,
}

## Why the helpers exist

The cells above show the quadratic phase and the resolution formulas directly so you can see the construction happen. After that first pass, those same operations move into the helper package so later notebooks can build waveforms and resolution numbers without repeating the derivation.

That is what the helper functions are for:

- they keep a single, correct waveform definition available to every later notebook,
- they reduce repeated code,
- and they make later lesson cells shorter once you already understand the idea.

As in Notebook 00, use the helpers when you want the lesson to stay readable, but keep the first occurrence of a concept visible in the notebook itself.

## Matched-filter compression preview

Pulse compression is what turns the long chirp into a sharp peak. The matched filter correlates the received signal with a time-reversed, conjugated copy of the transmit waveform. We preview it here with both the plain pulse and the chirp so the difference is visible: the plain pulse's filter output stays wide, while the chirp collapses to a narrow compressed peak.

In [ ]:
# Preview the matched-filter output for both waveforms.
from beginner.helpers import matched_filter

mf_rect = matched_filter(rect_pulse, rect_pulse)
mf_chirp = matched_filter(chirp, chirp)

# Zoom in around the peak so the width difference is clear.
peak = np.argmax(np.abs(mf_chirp))
span = pulse_samples
lo, hi = max(0, peak - span), min(len(mf_chirp), peak + span)
idx = np.arange(lo, hi)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(idx, np.abs(mf_rect[lo:hi]) / np.max(np.abs(mf_rect)), color="#d95f02", label="Rectangular pulse")
ax.plot(idx, np.abs(mf_chirp[lo:hi]) / np.max(np.abs(mf_chirp)), color="#1b9e77", label="LFM chirp")
ax.set_xlabel("Sample index around the compressed peak")
ax.set_ylabel("Normalized magnitude")
ax.set_title("Matched-filter output: chirp compresses, plain pulse stays wide")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("The chirp peak is narrow because its bandwidth, not its length, sets the width.")
print("Notebook 03 will turn this peak delay into a range estimate.")

## Closing the loop: answers to the opening questions

At the start of this notebook we listed four things you should be able to explain. Here is a direct answer to each one, using the physics, equations, and numbers we just worked through.

### What a rectangular pulse looks like in time

A rectangular pulse is simply the transmitter on at full amplitude for the pulse width, then off. In sampled form it is a block of ones, and its length is $N = f_s \cdot \tau$. With the baseline values that is $N = 20\ \text{MHz} \times 20\ \mu\text{s} = 400$ samples. There is no sweep, no hidden structure — just a flat burst of energy whose frequency content is limited to roughly $1/\tau = 50$ kHz. You saw this built directly in the first code cell and plotted as the top trace in the side-by-side figure.

### What an LFM chirp is and how its frequency sweeps

An LFM chirp keeps the same 400 samples and the same amplitude, but the complex phase now rises quadratically. The waveform is $s(t) = \exp\!\left(j \pi \frac{B}{\tau} t^{2}\right)$, with chirp rate $B/\tau = 2.5 \times 10^{11}$ Hz/s. The instantaneous frequency, visible in the second plot, ramps linearly from near zero up to approximately 5 MHz — the bandwidth — across the 20-microsecond pulse. That is the whole difference from the plain pulse: the frequency sweeps, and the sweep covers a band far wider than $1/\tau$. Everything that follows in this notebook hangs on that single fact.

### Why a chirp resolves targets much closer together than its pulse length suggests

Range resolution for a plain pulse is $\Delta R = c\tau/2 = 2997.9$ m. For a chirp after pulse compression it is $\Delta R = c/(2B) = 30.0$ m — a 100x improvement. The reason is visible in the matched-filter plot: the plain-pulse output stays wide, roughly 400 samples across, while the chirp collapses to a peak only a few samples wide. The compressed width is set by $1/B$, not by $\tau$. Because the bandwidth can be made far larger than the inverse pulse length, the chirp decouples energy from resolution — the pulse stays long for energy, the bandwidth stays wide for sharpness. The 100x factor you saw in the resolution comparison cell is the direct arithmetic of that decoupling.

### What the time-bandwidth product tells you about that gain

The time-bandwidth product is $TBP = B \cdot \tau = 5\ \text{MHz} \times 20\ \mu\text{s} = 100$. That number is the compression ratio: the long chirp, squeezed by the matched filter, produces a peak roughly 100 times narrower than the pulse itself. It measures exactly how much work the chirp is doing compared to a plain pulse of the same length. A large time-bandwidth product is what makes modern pulse radar practical — it lets the system stay loud and still resolve closely spaced targets. The TBP you computed in the cell above is the single number that captures the entire trade.

If you can retell these four answers in your own words — what the plain pulse is, how the chirp sweeps, why the bandwidth buys resolution, and what the time-bandwidth product means — you have the message of this notebook.

## Summary

In this notebook you went inside the transmit burst and built two waveforms. The rectangular pulse is simply on for the pulse width; its resolution is tied directly to that length. The LFM chirp keeps the same length but sweeps frequency across the bandwidth, and that sweep is what gives pulse compression its power.

The two formulas tell the story: $\Delta R = c\tau/2$ for the plain pulse, but $\Delta R = c/(2B)$ once the chirp is compressed. Because the bandwidth can be far larger than the inverse pulse length, a long, energetic chirp still resolves closely spaced targets. The time-bandwidth product captures exactly how much compression the waveform achieves.

The main takeaway is that radar resolution is a bandwidth story, not just a pulse-length story. The matched-filter preview at the end shows the practical result: the long chirp collapses into a sharp peak, which is the object Notebook 03 will later convert into range. For now, you should remember that the chirp lets the radar be both loud and sharp at the same time.